# Guardrail Config Development

## Prerequisites

Run `./setup_evalhub.sh` and select the **NeMo Guardrails Eval** kernel.

## Setup

We'll import first import the NeMoGuardrailsServer class from our `notebook_helpers` file. The NeMoGuardrailsServer is a simple class that does the following:

1) Initializes a local instance of a NeMo Guardrails server, by default on localhost:9999
2) Provides some useful functions : \
    a) `check(prompt)`: to send a specific prompt to the server's `/v1/guardrail/checks` endpoint \
    b) `logs()` which prints the server logs \
    c) `clean_up()` to shut down the server when we're done

So you can see what's happening, I've reproduced the `check` function below:

```python
    def check(self, prompt):
        r = requests.post(f"{self.base_url}/v1/guardrail/checks", json={
            "model": "dummy",
            "messages": [{"role": "user", "content": prompt}],
        })
        result = r.json()

        # remap a successful status to "allowed" for clarity
        if result["status"] == "success":
            result["status"] = "allowed"
        return result
```
As you can see, the `check` function is simply making a request to the server's `/v1/guardrail/checks` endpoint and then returning the result back to the user. 

## 1. Define the Guardrails Config

Edit the YAML below. To learn more about setting up and testing NeMo configurations, you can:
- reference the upstream [NeMo Guardrails documentation](https://docs.nvidia.com/nemo/guardrails/latest/configure-guardrails/overview).
- reference the [official Red Hat product documentation](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.5/html/enabling_ai_safety_with_guardrails/enabling-ai-safety-with-nemo-guardrails_nemo-guardrails#nemo-guardrails-standalone-quickstart_nemo-guardrails).
- check out the [local guardrail development demo notebook](./local_guardrail_development_demo.ipynb).

In [7]:
CONFIG_YAML = """\
rails:
  input:
    flows: []

  config: {}
"""

## 2. Start the NeMo Guardrails Server

In [8]:
from notebook_helpers import NeMoGuardrailsServer
guardrails_server = NeMoGuardrailsServer(CONFIG_YAML)

Server ready. Configs: [{'id': 'dev-config'}]


## 3. Send Queries

Test individual prompts against the running server using the `/v1/guardrail/checks` endpoint:

In [9]:
guardrails_server.check("Hi, how are you?")

{'status': 'allowed',
 'rails_status': {},
 'messages': [{'index': 0, 'role': 'user', 'rails': {}}],
 'guardrails_data': {'log': {'activated_rails': [],
   'stats': {'input_rails_duration': None,
    'dialog_rails_duration': None,
    'generation_rails_duration': None,
    'output_rails_duration': None,
    'total_duration': 0.005915164947509766,
    'llm_calls_duration': 0,
    'llm_calls_count': 0,
    'llm_calls_total_prompt_tokens': 0,
    'llm_calls_total_completion_tokens': 0,
    'llm_calls_total_tokens': 0}}}}

## 4. Cleanup

Stop the server and remove the temp config directory.

In [10]:
guardrails_server.cleanup()

Shutting down NeMo Guardrails server...[DONE]
